# UG-RAV on a Kaggle GPU — full-scale run

This notebook scales the CPU experiment in `scripts/run_real_experiment.py` to the configuration planned in the
dissertation: an open **7–8B instruction model in 4-bit**, **K = 10** samples, and **DeBERTa-large-MNLI** for
semantic clustering and evidence verification.

**Settings → Accelerator → GPU T4 ×2** (or P100) and **Internet → On**. A 4-bit 7B model fits in 16 GB.
Llama-3.1 requires accepting Meta's licence on the Hugging Face Hub and a token in *Add-ons → Secrets* (`HF_TOKEN`);
Qwen2.5 is open.

Expected cost: ≈ 25–40 s per question on a T4 for 22 generations + NLI → ~3 GPU-hours for 300 questions.
Every question is appended to a JSONL file, so the run can be resumed after a session restart.

In [ ]:
!git clone -q https://github.com/muditagarwal2308/HallucinationDetection.git
%cd HallucinationDetection
!pip install -q -e ".[ml]" bitsandbytes

In [ ]:
import os
try:  # Kaggle secret for gated models such as Llama-3.1
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

MODEL = "Qwen/Qwen2.5-7B-Instruct"     # or "meta-llama/Llama-3.1-8B-Instruct"
N_QUESTIONS = 300                       # stratified over PopQA popularity terciles
K = 10                                  # samples per question (closed-book and RAG)

In [ ]:
!python scripts/run_real_experiment.py --model {MODEL} --n {N_QUESTIONS} --k {K} --load-in-4bit --out results/real_gpu

## Results

The script prints the detection table (AUROC with bootstrap CIs) and the policy table, and writes
`results/real_gpu/benchmark_popqa_<model>.json`. Load it below, or copy it into `results/real/` of the repository
to see it in the Streamlit app (*UG-RAV Results → Real model*).

In [ ]:
import json, glob
from halludetect.ugrav.benchmark import detection_markdown, policy_markdown
from IPython.display import Markdown, display
res = json.load(open(sorted(glob.glob("results/real_gpu/benchmark_popqa_*.json"))[-1]))
display(Markdown(detection_markdown(res)))
display(Markdown(policy_markdown(res)))
print(json.dumps(res["popularity"], indent=1))

## Extending to TriviaQA and NQ-Open

`build_record(...)` in `halludetect.ugrav.signals` is dataset-agnostic: load any question set with gold aliases
(e.g. `mandarjoshi/trivia_qa`, config `rc.nocontext`, or `nq_open`), loop over it exactly as
`scripts/run_real_experiment.py` does, and pass the resulting records to `halludetect.ugrav.benchmark.evaluate`.